# aDDM Tutorial

This notebook showcases the implementation of a modern aDDM, compatible with PyDDM.

### Load the data

In [1]:
from ast import literal_eval
import pandas as pd

# 1. Load data
df_raw = pd.read_csv('1ms_trial_data.csv')

# 2. Drop nuisance trials
to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw.loc[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
    & (~df_raw["hidden"])
]

# 3. Adjustments
df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
df['choice'] = df['choice'].replace({"left": 0, "right": 1}) # Map choice to 0 or 1

/var/folders/8r/t10vclyx7wb951lg8h5g2jrd33843d/T/ipykernel_94419/4055317967.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
/var/folders/8r/t10vclyx7wb951lg8h5g2jrd33843d/T/ipykernel_94419/4055317967.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
/var/folders/8r/t10vclyx7wb951lg8h5g2jrd33843d/T/ipykernel_94419/4055317967.py:20: FutureWarning: Down

### Simulating data from empiricals

In [2]:
from simulation import get_corrected_empirical_distributions
import numpy as np

# Make empirical distributions
# value_diffs = np.arange(-4, 4.25, 0.25)
value_diffs = np.unique(df['avgWTP_left'] - df['avgWTP_right'])
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'

empirical_distributions = get_corrected_empirical_distributions(
    df,
    value_diffs=value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=0.9
)

In [3]:
from simulation import generate_fixations

# Create sample trial conditions
dt = 0.01
seed = 42

trials = df.loc[
    (df['sub_id'] == 304) & (df['trial'] % 2 == 1),
    ['avgWTP_left', 'avgWTP_right']
].copy()
trials['fixation'] = None

rng = np.random.default_rng(seed)
trials_dict = []
for idx, r in trials.iterrows():
    fx = generate_fixations(
        dt, 
        r.avgWTP_left - r.avgWTP_right, 
        empirical_distributions,
        rng=rng
    )
    if fx is not None:
        trials_dict.append({
            "avgWTP_left": r.avgWTP_left,
            "avgWTP_right": r.avgWTP_right,
            "fixation": fx
        })

In [4]:
from simulation import simulate
import pyddm

model_conditions = {'drift_rate': 0.3, 'theta': 0.5, 'noise': 0.7}

results_df = simulate(dt, model_conditions, trials_dict, seed=seed, save_results=False)
# results_df['sub_id'] = f'seed{seed}_subjects{size}_sim'
# results_df['trial'] = range(1, len(trials_clean) + 1)
# results_df = results_df.rename(columns={'fixation': 'fix_sequence'})
results_df = results_df.drop(columns = ['trajectory'])

sample = pyddm.Sample.from_pandas_dataframe(
    results_df,
    choice_column_name="choice",
    rt_column_name="RT",
    choice_names=("left", "right")
)

print(f'Average RT: {results_df["RT"].mean():.2f} seconds (out of {len(results_df)} trials)')
results_df.head()

Average RT: 1.80 seconds (out of 100 trials)


,avgWTP_left,avgWTP_right,fixation,RT,choice
0,1.00,1.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3.19,1
1,4.25,3.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",4.52,1
2,3.25,3.75,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, ...",2.12,0
3,3.00,2.75,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",5.29,1
4,1.00,4.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.38,0


The above is the first half of the tutorial. Following is native parameter recovery by differential evolution.

In [5]:
import numpy as np

# Define the model
def drift_function(avgWTP_left, avgWTP_right, fixation, d, x, t):
        fixation_index = min(int(t/dt), len(fixation)-1)
        current_fixation = fixation[fixation_index]
        if current_fixation == 0: # saccade
            drift_val = 0
        elif current_fixation == 1: # left
            drift_val = d * (avgWTP_left - avgWTP_right * model_conditions['theta'])
        else: # right
            drift_val = d * (avgWTP_left * model_conditions['theta'] - avgWTP_right)

        return np.ones_like(x) * drift_val

def noise_function(n, x, t):
    return np.ones_like(x) * n

model = pyddm.gddm(
    drift=drift_function,
    noise=noise_function,
    bound=1,
    nondecision=0,
    parameters={'d': (0.15, 0.45), 'n': (0.5, 0.9)},
    conditions=["avgWTP_left", "avgWTP_right", "fixation"],
    choice_names=("left", "right"),
    T_dur=30,
    dx=0.01,
    dt=dt
)

model._overlay = pyddm.models.OverlayChain(overlays=[])

model.fit(sample=sample, verbose=True)

Info: Model(name='n', drift=DriftEasy(d=Fitted(0.2475485421219057, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7766139507429481, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=261.84842185121005
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.390297153590641, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.6482259254416295, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=330.0528332738325
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.31372193180469055, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.8523265095506087, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=inf
Info: Model(na

differential_evolution step 1: f(x)= 241.5004070035145


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.3409095435706052, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7297250637904733, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=291.1103777707547
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.21051742161563713, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7878034912976954, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=253.30406629052555
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.1713728541886237, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.8462813053293679, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=246.819017457079


differential_evolution step 2: f(x)= 241.5004070035145


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.20030140473392483, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.8853865028816471, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=inf
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.25524234636880494, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.87288894811288, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=inf
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.40103661055498097, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7471897090191146, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=305.72519558278026
Info: Model(name='n', drift

...

Info: Model(name='n', drift=DriftEasy(d=Fitted(0.1624399230520303, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7768165672189733, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=243.78262749493268
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.3918866348495879, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.749697877982195, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=302.5536494790561
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.3639934618618137, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.8226159773931989, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=287.5740991329209
I

differential_evolution step 6: f(x)= 241.316452277337


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.34797988604562846, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7026342334740145, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=298.766023037835
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.19186232091509386, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.830966534604147, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=249.84896806069895
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.1713728541886237, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.798232921215048, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=245.33205622883963


differential_evolution step 7: f(x)= 241.2832118292638
Polishing solution with 'L-BFGS-B'


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.15005306737823956, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7929943979673316, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=241.2832118292638
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.15005307737823956, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7929943979673316, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=241.2832136901133
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.15005306737823956, minval=0.15, maxval=0.45)), noise=NoiseEasy(n=Fitted(0.7929944079673317, minval=0.5, maxval=0.9)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=241.283211869313